In [1]:
"""
Decoding Analysis: Spatial vs. Temporal Clustering
====================================================
Binary decoding only (strict labels)
  Label  : spatial_label==1 & temporal_label==0  vs
           temporal_label==1 & spatial_label==0
  Trials : ~51 trials (strict subset)
  CV     : Leave-One-Subject-Out (LOSO)
           (within-subject CV not feasible: ~1-2 trials per class per subject)

Features : encoding or retrieval epoch
           frac_ssl + osc_ssl at each of 18 freqs × 2 hemispheres (LHP+RHP)
           → 2 DVs × 18 freqs × 2 hemispheres = 72 features per trial
Permutation test: 1000 iterations, labels shuffled within subject

Outputs:
  decoding_binary_summary.csv
  decoding_binary_per_subject.csv
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV    = "ALL_SUBJECTS_irasa_labeled.csv"
N_PERM       = 1000
RANDOM_STATE = 42
TRIAL_ID     = ["subject", "session", "trial"]

MODELS = {
    "encoding":  ["encoding_frac_ssl",  "encoding_osc_ssl"],
    "retrieval": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
}

# ── Load ──────────────────────────────────────────────────────────────────────
print("Loading data ...")
df = pd.read_csv(INPUT_CSV)
df = df[df["region"].isin(["LHP", "RHP"])].copy()

# ── Cluster label (binary) ────────────────────────────────────────────────────
def assign_cluster(row):
    if row["spatial_label"] == 1 and row["temporal_label"] == 0:
        return "spatial"
    elif row["temporal_label"] == 1 and row["spatial_label"] == 0:
        return "temporal"
    else:
        return "neither"

df["cluster_type"] = df.apply(assign_cluster, axis=1)

# ── Build wide feature matrix ─────────────────────────────────────────────────
def build_feature_matrix(df_all, dv_list):
    """
    Pivot long -> wide with LHP and RHP as feature dimensions.
    Keeps only strict spatial/temporal trials.
    Multiple electrodes averaged within trial/region/freq.
    """
    d = df_all[df_all["cluster_type"].isin(["spatial", "temporal"])].copy()
    d["freq_tag"] = d["freq_hz"].round(2).astype(str)

    hemi_parts = []
    for hemi in ["LHP", "RHP"]:
        dh = d[d["region"] == hemi].copy()
        parts = []
        for dv in dv_list:
            piv = dh.pivot_table(
                index   = TRIAL_ID + ["cluster_type"],
                columns = "freq_tag",
                values  = dv,
                aggfunc = "mean"
            )
            piv.columns = [f"{hemi}_{dv}@{c}Hz" for c in piv.columns]
            parts.append(piv)
        hemi_parts.append(pd.concat(parts, axis=1))

    wide = hemi_parts[0].join(hemi_parts[1], how="inner").reset_index()
    wide = wide.dropna()
    return wide

def get_feature_cols(wide, dv_list):
    return [c for c in wide.columns if any(f"_{dv}@" in c for dv in dv_list)]

# ════════════════════════════════════════════════════════════════════════════════
# LOSO Binary Decoding
# ════════════════════════════════════════════════════════════════════════════════

def loso_decode_binary(wide, feature_cols):
    subjects = wide["subject"].unique()
    y_true_all, y_prob_all, y_pred_all, subj_ids = [], [], [], []

    for subj in subjects:
        train = wide[wide["subject"] != subj]
        test  = wide[wide["subject"] == subj]

        if test["cluster_type"].nunique() < 2:
            print(f"    Skipping {subj} (only one class in test set)")
            continue

        X_tr = train[feature_cols].values
        y_tr = (train["cluster_type"] == "spatial").astype(int).values
        X_te = test[feature_cols].values
        y_te = (test["cluster_type"] == "spatial").astype(int).values

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(C=1.0, solver="liblinear",
                                          max_iter=1000, random_state=RANDOM_STATE))
        ])
        pipe.fit(X_tr, y_tr)
        y_prob_all.extend(pipe.predict_proba(X_te)[:, 1])
        y_pred_all.extend(pipe.predict(X_te))
        y_true_all.extend(y_te)
        subj_ids.extend([subj] * len(y_te))

    y_true = np.array(y_true_all)
    y_prob = np.array(y_prob_all)
    y_pred = np.array(y_pred_all)

    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    per_subj_df = pd.DataFrame({
        "subject":    subj_ids,
        "true_label": y_true,
        "pred_label": y_pred,
        "pred_prob":  y_prob,
    })
    return auc, acc, per_subj_df


def perm_binary(wide, true_auc, feature_cols):
    subjects  = wide["subject"].unique()
    rng       = np.random.default_rng(RANDOM_STATE)
    perm_aucs = []

    for i in range(N_PERM):
        wp = wide.copy()
        wp["cluster_type"] = (
            wp.groupby("subject")["cluster_type"]
              .transform(lambda x: x.sample(frac=1,
                         random_state=int(rng.integers(1e6))).values)
        )
        y_true_p, y_prob_p = [], []
        for subj in subjects:
            train = wp[wp["subject"] != subj]
            test  = wp[wp["subject"] == subj]
            if test["cluster_type"].nunique() < 2:
                continue
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(C=1.0, solver="liblinear",
                           max_iter=1000, random_state=RANDOM_STATE))
            ])
            pipe.fit(train[feature_cols].values,
                     (train["cluster_type"] == "spatial").astype(int).values)
            y_true_p.extend((test["cluster_type"] == "spatial").astype(int).values)
            y_prob_p.extend(pipe.predict_proba(test[feature_cols].values)[:, 1])
        try:
            perm_aucs.append(roc_auc_score(np.array(y_true_p), np.array(y_prob_p)))
        except Exception:
            pass
        if (i + 1) % 100 == 0:
            print(f"    permutation {i+1}/{N_PERM} ...")

    perm_aucs = np.array(perm_aucs)
    p_val = (np.sum(perm_aucs >= true_auc) + 1) / (len(perm_aucs) + 1)
    return p_val, perm_aucs


def sig_stars(p):
    return "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))

# ════════════════════════════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════════════════════════════

summary_rows   = []
per_subj_parts = []

for model_name, dv_list in MODELS.items():
    print(f"\n{'='*60}")
    print(f"[BINARY] Model: {model_name}")

    wide         = build_feature_matrix(df, dv_list)
    feature_cols = get_feature_cols(wide, dv_list)

    n_trials   = len(wide)
    n_subjects = wide["subject"].nunique()
    print(f"  Trials: {n_trials}  |  Subjects: {n_subjects}  |  Features: {len(feature_cols)}")
    print(f"  Class balance: {wide['cluster_type'].value_counts().to_dict()}")

    auc, acc, per_subj_df = loso_decode_binary(wide, feature_cols)
    per_subj_df["model"] = model_name
    print(f"  LOSO AUC = {auc:.4f}   Accuracy = {acc:.4f}")

    print(f"  Running {N_PERM} permutations ...")
    p_val, perm_aucs = perm_binary(wide, auc, feature_cols)
    print(f"  Permutation p = {p_val:.4f}  {sig_stars(p_val)}")

    summary_rows.append({
        "model":       model_name,
        "n_trials":    n_trials,
        "n_subjects":  n_subjects,
        "n_features":  len(feature_cols),
        "auc":         round(auc, 4),
        "accuracy":    round(acc, 4),
        "perm_p":      round(p_val, 4),
        "sig":         sig_stars(p_val),
    })
    per_subj_parts.append(per_subj_df)

# ── Save ──────────────────────────────────────────────────────────────────────
pd.DataFrame(summary_rows).to_csv("decoding_binary_summary.csv", index=False)
pd.concat(per_subj_parts).to_csv("decoding_binary_per_subject.csv", index=False)

print("\n\n-- Binary Summary -------------------------------------------")
print(pd.DataFrame(summary_rows).to_string(index=False))
print("\nSaved: decoding_binary_summary.csv, decoding_binary_per_subject.csv")

Loading data ...

[BINARY] Model: encoding
  Trials: 51  |  Subjects: 15  |  Features: 72
  Class balance: {'temporal': 36, 'spatial': 15}
    Skipping R1520T (only one class in test set)
    Skipping R1521E (only one class in test set)
    Skipping R1523E (only one class in test set)
    Skipping R1532T (only one class in test set)
    Skipping R1551T (only one class in test set)
    Skipping R1570T (only one class in test set)
    Skipping R1642J (only one class in test set)
    Skipping R1653J (only one class in test set)
  LOSO AUC = 0.5909   Accuracy = 0.6471
  Running 1000 permutations ...
    permutation 100/1000 ...
    permutation 200/1000 ...
    permutation 300/1000 ...
    permutation 400/1000 ...
    permutation 500/1000 ...
    permutation 600/1000 ...
    permutation 700/1000 ...
    permutation 800/1000 ...
    permutation 900/1000 ...
    permutation 1000/1000 ...
  Permutation p = 0.0689  ns

[BINARY] Model: retrieval
  Trials: 51  |  Subjects: 15  |  Features: 72
  C

In [8]:
"""
Decoding Analysis: Spatial Label and Temporal Label (Separate)
===============================================================
Two independent binary decodings:
  1. Spatial  : spatial_label==1  vs  spatial_label==0
  2. Temporal : temporal_label==1 vs  temporal_label==0

  CV     : Leave-One-Subject-Out (LOSO)
Features : encoding or retrieval epoch
           frac_ssl + osc_ssl at each of 18 freqs × 2 hemispheres (LHP+RHP)
           → 2 DVs × 18 freqs × 2 hemispheres = 72 features per trial
Permutation test: 1000 iterations, labels shuffled within subject

Outputs:
  decoding_separate_summary.csv
  decoding_separate_per_subject.csv
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV    = "ALL_SUBJECTS_irasa_labeled.csv"
N_PERM       = 1000
RANDOM_STATE = 42
TRIAL_ID     = ["subject", "session", "trial", "recalled_word"]

MODELS = {
    "encoding":  ["encoding_frac_ssl",  "encoding_osc_ssl"],
    "retrieval": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
}

# Two separate decoding targets
DECODE_TARGETS = {
    "spatial":  "spatial_label",
    "temporal": "temporal_label",
}

# ── Load ──────────────────────────────────────────────────────────────────────
print("Loading data ...")
df = pd.read_csv(INPUT_CSV)
df = df[df["region"].isin(["LHP", "RHP"])].copy()

# ── Build wide feature matrix ─────────────────────────────────────────────────
def build_feature_matrix(df_all, dv_list, label_col):
    """
    Pivot long -> wide. Uses all trials (label 0 or 1 for the given label_col).
    Multiple electrodes averaged within trial/region/freq.
    """
    d = df_all.copy()
    d["freq_tag"] = d["freq_hz"].round(2).astype(str)

    hemi_parts = []
    for hemi in ["LHP", "RHP"]:
        dh = d[d["region"] == hemi].copy()
        parts = []
        for dv in dv_list:
            piv = dh.pivot_table(
                index   = TRIAL_ID + [label_col],
                columns = "freq_tag",
                values  = dv,
                aggfunc = "mean"
            )
            piv.columns = [f"{hemi}_{dv}@{c}Hz" for c in piv.columns]
            parts.append(piv)
        hemi_parts.append(pd.concat(parts, axis=1))

    wide = hemi_parts[0].join(hemi_parts[1], how="inner").reset_index()
    wide = wide.dropna()
    return wide

def get_feature_cols(wide, dv_list):
    return [c for c in wide.columns if any(f"_{dv}@" in c for dv in dv_list)]

# ════════════════════════════════════════════════════════════════════════════════
# LOSO Binary Decoding
# ════════════════════════════════════════════════════════════════════════════════

def loso_decode_binary(wide, feature_cols, label_col):
    subjects = wide["subject"].unique()
    y_true_all, y_prob_all, y_pred_all, subj_ids = [], [], [], []

    for subj in subjects:
        train = wide[wide["subject"] != subj]
        test  = wide[wide["subject"] == subj]

        if test[label_col].nunique() < 2:
            print(f"    Skipping {subj} (only one class in test set)")
            continue

        X_tr = train[feature_cols].values
        y_tr = train[label_col].values
        X_te = test[feature_cols].values
        y_te = test[label_col].values

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(C=1.0, solver="liblinear",
                                          max_iter=1000, random_state=RANDOM_STATE))
        ])
        pipe.fit(X_tr, y_tr)
        y_prob_all.extend(pipe.predict_proba(X_te)[:, 1])
        y_pred_all.extend(pipe.predict(X_te))
        y_true_all.extend(y_te)
        subj_ids.extend([subj] * len(y_te))

    y_true = np.array(y_true_all)
    y_prob = np.array(y_prob_all)
    y_pred = np.array(y_pred_all)

    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, y_pred)
    per_subj_df = pd.DataFrame({
        "subject":    subj_ids,
        "true_label": y_true,
        "pred_label": y_pred,
        "pred_prob":  y_prob,
    })
    return auc, acc, per_subj_df


def perm_binary(wide, true_auc, feature_cols, label_col):
    subjects  = wide["subject"].unique()
    rng       = np.random.default_rng(RANDOM_STATE)
    perm_aucs = []

    for i in range(N_PERM):
        wp = wide.copy()
        wp[label_col] = (
            wp.groupby("subject")[label_col]
              .transform(lambda x: x.sample(frac=1,
                         random_state=int(rng.integers(1e6))).values)
        )
        y_true_p, y_prob_p = [], []
        for subj in subjects:
            train = wp[wp["subject"] != subj]
            test  = wp[wp["subject"] == subj]
            if test[label_col].nunique() < 2:
                continue
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(C=1.0, solver="liblinear",
                           max_iter=1000, random_state=RANDOM_STATE))
            ])
            pipe.fit(train[feature_cols].values, train[label_col].values)
            y_true_p.extend(test[label_col].values)
            y_prob_p.extend(pipe.predict_proba(test[feature_cols].values)[:, 1])
        try:
            perm_aucs.append(roc_auc_score(np.array(y_true_p), np.array(y_prob_p)))
        except Exception:
            pass
        if (i + 1) % 100 == 0:
            print(f"    permutation {i+1}/{N_PERM} ...")

    perm_aucs = np.array(perm_aucs)
    p_val = (np.sum(perm_aucs >= true_auc) + 1) / (len(perm_aucs) + 1)
    return p_val, perm_aucs


def sig_stars(p):
    return "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))

# ════════════════════════════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════════════════════════════

summary_rows   = []
per_subj_parts = []

for decode_name, label_col in DECODE_TARGETS.items():
    for model_name, dv_list in MODELS.items():
        print(f"\n{'='*60}")
        print(f"[{decode_name.upper()}] Model: {model_name}  |  Label: {label_col}")

        wide         = build_feature_matrix(df, dv_list, label_col)
        feature_cols = get_feature_cols(wide, dv_list)

        n_trials   = len(wide)
        n_subjects = wide["subject"].nunique()
        class_bal  = wide[label_col].value_counts().to_dict()
        print(f"  Trials: {n_trials}  |  Subjects: {n_subjects}  |  Features: {len(feature_cols)}")
        print(f"  Class balance (0 vs 1): {class_bal}")

        auc, acc, per_subj_df = loso_decode_binary(wide, feature_cols, label_col)
        per_subj_df["decode_target"] = decode_name
        per_subj_df["model"]         = model_name
        print(f"  LOSO AUC = {auc:.4f}   Accuracy = {acc:.4f}")

        print(f"  Running {N_PERM} permutations ...")
        p_val, perm_aucs = perm_binary(wide, auc, feature_cols, label_col)
        print(f"  Permutation p = {p_val:.4f}  {sig_stars(p_val)}")

        summary_rows.append({
            "decode_target": decode_name,
            "model":         model_name,
            "label_col":     label_col,
            "n_trials":      n_trials,
            "n_subjects":    n_subjects,
            "n_features":    len(feature_cols),
            "n_label_1":     class_bal.get(1, 0),
            "n_label_0":     class_bal.get(0, 0),
            "auc":           round(auc, 4),
            "accuracy":      round(acc, 4),
            "perm_p":        round(p_val, 4),
            "sig":           sig_stars(p_val),
        })
        per_subj_parts.append(per_subj_df)

# ── Save ──────────────────────────────────────────────────────────────────────
pd.DataFrame(summary_rows).to_csv("decoding_separate_summary.csv", index=False)
pd.concat(per_subj_parts).to_csv("decoding_separate_per_subject.csv", index=False)

print("\n\n-- Summary -------------------------------------------")
print(pd.DataFrame(summary_rows).to_string(index=False))
print("\nSaved: decoding_separate_summary.csv, decoding_separate_per_subject.csv")

Loading data ...

[SPATIAL] Model: encoding  |  Label: spatial_label
  Trials: 508  |  Subjects: 18  |  Features: 72
  Class balance (0 vs 1): {0: 489, 1: 19}
    Skipping R1520T (only one class in test set)
    Skipping R1521E (only one class in test set)
    Skipping R1523E (only one class in test set)
    Skipping R1532T (only one class in test set)
    Skipping R1536J (only one class in test set)
    Skipping R1551T (only one class in test set)
    Skipping R1569T (only one class in test set)
    Skipping R1570T (only one class in test set)
    Skipping R1572T (only one class in test set)
    Skipping R1573T (only one class in test set)
    Skipping R1575E (only one class in test set)
    Skipping R1642J (only one class in test set)
    Skipping R1651J (only one class in test set)
    Skipping R1653J (only one class in test set)
  LOSO AUC = 0.3699   Accuracy = 0.8545
  Running 1000 permutations ...
    permutation 100/1000 ...
    permutation 200/1000 ...
    permutation 300/1000 

In [9]:
"""
Within-Subject Neural–Behavior Correlation Analysis
=====================================================
Approach (standard in iEEG/memory neuroscience):
  1. For each subject × freq × region × epoch:
       Compute Spearman r between neural feature (frac_ssl or osc_ssl)
       and continuous clustering score (SC_t or TC_t)
  2. Fisher z-transform the r values
  3. Group-level test: one-sample t-test (or Wilcoxon) of z-values against 0
     across subjects → one p-value per freq × region × epoch × behavior

This is analogous to Jacobs et al. / Manning et al. style single-trial
neural–behavior correlation, and is consistent with the LME framework
already used in this manuscript.

FDR correction (Benjamini-Hochberg) across frequencies within each
condition (epoch × region × dv × behavior).

Outputs:
  within_subj_corr_summary.csv   — group-level stats per freq × condition
  within_subj_corr_persubj.csv   — per-subject r values (for raincloud plots)
"""

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
INPUT_CSV  = "ALL_SUBJECTS_irasa_labeled.csv"
TRIAL_ID   = ["subject", "session", "trial", "recalled_word"]
REGIONS    = ["LHP", "RHP"]
MIN_WORDS  = 10   # minimum words per subject to include in group test

EPOCHS = {
    "encoding":  ["encoding_frac_ssl",  "encoding_osc_ssl"],
    "retrieval": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
}

BEHAVIORS = {
    "spatial":  "SC_t",
    "temporal": "TC_t",
}

# ── Load & clean ───────────────────────────────────────────────────────────────
print("Loading data ...")
df = pd.read_csv(INPUT_CSV)
df = df[df["region"].isin(REGIONS)].copy()

# Remove inf values in behavior scores
for col in ["SC_t", "TC_t"]:
    n_inf = np.isinf(df[col]).sum()
    if n_inf > 0:
        print(f"  Removing {n_inf} rows with inf in {col}")
    df = df[~np.isinf(df[col])]

print(f"  Subjects: {df['subject'].nunique()}")
print(f"  Total rows: {len(df)}")

# ── Per-subject Spearman r ─────────────────────────────────────────────────────
def compute_within_subj_corr(df, dv, behavior_col, region):
    """
    For each subject × freq: Spearman r between dv and behavior_col.
    Returns DataFrame with columns [subject, freq_hz, r, n].
    """
    rows = []
    d = df[df["region"] == region].copy()

    for subj, sg in d.groupby("subject"):
        for freq, fg in sg.groupby("freq_hz"):
            # One row per word — average across electrodes within subj/freq/word
            word_avg = fg.groupby(TRIAL_ID)[
                [dv, behavior_col]
            ].mean().reset_index()

            word_avg = word_avg.dropna(subset=[dv, behavior_col])
            n = len(word_avg)
            if n < MIN_WORDS:
                continue

            r, _ = stats.spearmanr(word_avg[dv], word_avg[behavior_col])
            rows.append({
                "subject": subj,
                "freq_hz": freq,
                "r":       r,
                "n":       n,
            })

    return pd.DataFrame(rows)


# ── Group-level test (Fisher z → one-sample t) ────────────────────────────────
def group_test(corr_df):
    """
    For each freq: Fisher-z transform r values, one-sample t-test vs 0.
    Returns DataFrame with group-level stats per freq.
    """
    rows = []
    for freq, fg in corr_df.groupby("freq_hz"):
        r_vals = fg["r"].values
        # Fisher z-transform (clip r away from ±1 to avoid inf)
        r_clipped = np.clip(r_vals, -0.9999, 0.9999)
        z_vals    = np.arctanh(r_clipped)
        n_subj    = len(z_vals)

        if n_subj < 3:
            continue

        t, p_t   = stats.ttest_1samp(z_vals, 0)
        w, p_w   = stats.wilcoxon(z_vals) if n_subj >= 5 else (np.nan, np.nan)
        mean_r   = np.tanh(np.mean(z_vals))   # back-transform mean z → r
        sem_z    = np.std(z_vals, ddof=1) / np.sqrt(n_subj)

        rows.append({
            "freq_hz":  freq,
            "n_subj":   n_subj,
            "mean_r":   round(mean_r, 4),
            "mean_z":   round(np.mean(z_vals), 4),
            "sem_z":    round(sem_z, 4),
            "t":        round(t, 4),
            "p_ttest":  round(p_t, 4),
            "w_stat":   round(w, 4) if not np.isnan(w) else np.nan,
            "p_wilcox": round(p_w, 4) if not np.isnan(p_w) else np.nan,
        })

    result = pd.DataFrame(rows).sort_values("freq_hz")

    # FDR correction across frequencies
    if len(result) > 0:
        _, fdr_p, _, _ = multipletests(result["p_ttest"], method="fdr_bh")
        result["p_fdr"] = fdr_p.round(4)
        result["sig"]   = result["p_fdr"].apply(
            lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        )

    return result


# ════════════════════════════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════════════════════════════

summary_parts  = []
persubj_parts  = []

for beh_name, beh_col in BEHAVIORS.items():
    for epoch_name, dv_list in EPOCHS.items():
        for dv in dv_list:
            for region in REGIONS:
                label = f"{beh_name} | {epoch_name} | {dv} | {region}"
                print(f"\n{'─'*60}")
                print(f"  {label}")

                corr_df = compute_within_subj_corr(df, dv, beh_col, region)
                if corr_df.empty:
                    print("  No data — skipping")
                    continue

                group_df = group_test(corr_df)

                # Tag columns
                for d_ in [corr_df, group_df]:
                    d_["behavior"]  = beh_name
                    d_["epoch"]     = epoch_name
                    d_["dv"]        = dv
                    d_["region"]    = region

                # Print significant freqs
                sig_rows = group_df[group_df.get("p_fdr", group_df["p_ttest"]) < 0.05]
                if len(sig_rows):
                    print(f"  Significant freqs (FDR < 0.05):")
                    print(sig_rows[["freq_hz","mean_r","t","p_ttest","p_fdr","sig"]].to_string(index=False))
                else:
                    print("  No significant freqs after FDR correction")

                summary_parts.append(group_df)
                persubj_parts.append(corr_df)

# ── Save ──────────────────────────────────────────────────────────────────────
summary_df = pd.concat(summary_parts, ignore_index=True)
persubj_df = pd.concat(persubj_parts, ignore_index=True)

summary_df.to_csv("within_subj_corr_summary.csv", index=False)
persubj_df.to_csv("within_subj_corr_persubj.csv",  index=False)

print("\n\n══ FULL SUMMARY ══════════════════════════════════════════════")
cols_show = ["behavior","epoch","dv","region","freq_hz","n_subj","mean_r","t","p_ttest","p_fdr","sig"]
print(summary_df[cols_show].to_string(index=False))
print("\nSaved: within_subj_corr_summary.csv, within_subj_corr_persubj.csv")

Loading data ...
  Removing 54 rows with inf in SC_t
  Removing 522 rows with inf in TC_t
  Subjects: 36
  Total rows: 28260

────────────────────────────────────────────────────────────
  spatial | encoding | encoding_frac_ssl | LHP
  No significant freqs after FDR correction

────────────────────────────────────────────────────────────
  spatial | encoding | encoding_frac_ssl | RHP
  No significant freqs after FDR correction

────────────────────────────────────────────────────────────
  spatial | encoding | encoding_osc_ssl | LHP
  No significant freqs after FDR correction

────────────────────────────────────────────────────────────
  spatial | encoding | encoding_osc_ssl | RHP
  No significant freqs after FDR correction

────────────────────────────────────────────────────────────
  spatial | retrieval | retrieval_frac_ssl | LHP
  No significant freqs after FDR correction

────────────────────────────────────────────────────────────
  spatial | retrieval | retrieval_frac_ssl | RHP

In [12]:
"""
Elastic-Net Regression: Band-Averaged Neural Features → Word-Level Clustering
==============================================================================
Unit of analysis : WORD (session pooled within subject)

Feature reduction: 18 freqs averaged into 5 canonical bands
  theta     :  3.00 –  9.05 Hz  (3.00, 3.74, 4.67, 5.82, 7.26)
  alpha     :  9.05 – 14.07 Hz  (9.05, 11.28)
  beta      : 14.07 – 34.03 Hz  (14.07, 17.55, 21.88, 27.29)
  low_gamma : 34.03 – 66.00 Hz  (34.03, 42.44, 52.92)
  high_gamma: 66.00 – 128.0 Hz  (66.00, 82.31, 102.64, 128.0)

Features per word: 5 bands × 2 regions (LHP, RHP) × 2 DVs (frac_ssl, osc_ssl)
  = 20 features total

Outcomes (4, run separately):
  SP_clustering_from_prev, SP_clustering_to_next
  T_clustering_from_prev,  T_clustering_to_next

Epochs (2, run separately):
  encoding, retrieval

Model     : ElasticNetCV within each subject (sessions pooled)
Validation: Leave-One-Out CV → Spearman r(y, y_hat)
Permutation: 1000 shuffles of y within subject → subject-level p-value
Group test: Fisher z → one-sample t-test + Wilcoxon across subjects
FDR       : Benjamini-Hochberg across 8 conditions (4 outcomes × 2 epochs)

Outputs:
  elasticnet_band_summary.csv       — group-level stats
  elasticnet_band_persubj.csv       — per-subject LOO r + perm p
  elasticnet_band_coefficients.csv  — mean coefficients (feature importance)
"""

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────────
INPUT_CSV    = "ALL_SUBJECTS_irasa_clean.csv"
N_PERM       = 1000
RANDOM_STATE = 42
MIN_WORDS    = 20
REGIONS      = ["LHP", "RHP"]
TRIAL_ID     = ["subject", "session", "trial", "recalled_word"]

FREQ_BANDS = {
    "theta":      [3.00, 3.74, 4.67, 5.82, 7.26],
    "alpha":      [9.05, 11.28],
    "beta":       [14.07, 17.55, 21.88, 27.29],
    "low_gamma":  [34.03, 42.44, 52.92],
    "high_gamma": [66.00, 82.31, 102.64, 128.0],
}

EPOCHS = {
    "encoding":  ["encoding_frac_ssl",  "encoding_osc_ssl"],
    "retrieval": ["retrieval_frac_ssl", "retrieval_osc_ssl"],
}

OUTCOMES = [
    "SP_clustering_from_prev",
    "SP_clustering_to_next",
    "T_clustering_from_prev",
    "T_clustering_to_next",
]

L1_RATIOS = [0.1, 0.5, 0.7, 0.9, 0.95, 1.0]
ALPHAS    = np.logspace(-4, 2, 30)   # wider range than before

# ── Load ───────────────────────────────────────────────────────────────────────
print("Loading data ...")
df = pd.read_csv(INPUT_CSV)
df = df[df["region"].isin(REGIONS)].copy()
print(f"  Subjects : {df['subject'].nunique()}")
print(f"  Rows     : {len(df)}")

# ── Band averaging ─────────────────────────────────────────────────────────────
def add_band_label(df):
    """Assign each row a band label based on freq_hz."""
    freq_to_band = {}
    for band, freqs in FREQ_BANDS.items():
        for f in freqs:
            freq_to_band[round(f, 2)] = band
    df = df.copy()
    df["band"] = df["freq_hz"].round(2).map(freq_to_band)
    df = df[df["band"].notna()]   # drop any freqs not in bands
    return df

df = add_band_label(df)
print(f"  Band counts:\n{df['band'].value_counts().to_string()}")

# ── Build word-level band-averaged feature matrix ─────────────────────────────
def build_band_matrix(df, dv_list, outcome_col):
    """
    For each word:
      1. Average neural features within band × region
      2. Pivot to wide: one column per band × region × dv
    Returns wide DataFrame, one row per word.
    """
    # Average across freqs within band, and across electrodes within region
    band_avg = (
        df.groupby(TRIAL_ID + [outcome_col, "region", "band"])[dv_list]
        .mean()
        .reset_index()
    )

    # Pivot wide: columns = region_dv_band
    hemi_parts = []
    for hemi in REGIONS:
        dh = band_avg[band_avg["region"] == hemi]
        parts = []
        for dv in dv_list:
            piv = dh.pivot_table(
                index   = TRIAL_ID + [outcome_col],
                columns = "band",
                values  = dv,
                aggfunc = "mean"
            )
            piv.columns = [f"{hemi}_{dv}_{c}" for c in piv.columns]
            parts.append(piv)
        hemi_parts.append(pd.concat(parts, axis=1))

    wide = hemi_parts[0].join(hemi_parts[1], how="inner").reset_index()
    wide = wide.dropna()   # removes first/last word of each list (NaN outcome)
    return wide

def get_feature_cols(wide, dv_list):
    return [c for c in wide.columns
            if any(f"_{dv}_" in c for dv in dv_list)]

# ── LOO elastic-net ────────────────────────────────────────────────────────────
def loo_elasticnet(X, y):
    n = len(y)

    # Fit on full data to select hyperparams
    pipe_full = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  ElasticNetCV(
            l1_ratio     = L1_RATIOS,
            alphas       = ALPHAS,
            cv           = min(5, n - 1),
            max_iter     = 10000,
            random_state = RANDOM_STATE
        ))
    ])
    pipe_full.fit(X, y)
    best_alpha = pipe_full.named_steps["model"].alpha_
    best_l1    = pipe_full.named_steps["model"].l1_ratio_
    coefs      = pipe_full.named_steps["model"].coef_
    n_nonzero  = int(np.sum(coefs != 0))

    # LOO predictions with fixed hyperparams
    loo    = LeaveOneOut()
    y_pred = np.zeros(n)
    for train_idx, test_idx in loo.split(X):
        pipe_loo = Pipeline([
            ("scaler", StandardScaler()),
            ("model",  ElasticNet(
                alpha    = best_alpha,
                l1_ratio = best_l1,
                max_iter = 10000,
            ))
        ])
        pipe_loo.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = pipe_loo.predict(X[test_idx])

    r, _ = stats.spearmanr(y, y_pred)
    return r, best_alpha, best_l1, coefs, n_nonzero

# ── Permutation test ───────────────────────────────────────────────────────────
def permutation_test(X, y, true_r, best_alpha, best_l1):
    """Shuffle y, rerun LOO with fixed hyperparams, build null distribution."""
    loo     = LeaveOneOut()
    rng     = np.random.default_rng(RANDOM_STATE)
    perm_rs = []
    n       = len(y)

    for _ in range(N_PERM):
        y_shuf = rng.permutation(y)
        y_pred = np.zeros(n)
        for train_idx, test_idx in loo.split(X):
            pipe_p = Pipeline([
                ("scaler", StandardScaler()),
                ("model",  ElasticNet(
                    alpha    = best_alpha,
                    l1_ratio = best_l1,
                    max_iter = 10000,
                ))
            ])
            pipe_p.fit(X[train_idx], y_shuf[train_idx])
            y_pred[test_idx] = pipe_p.predict(X[test_idx])
        r_p, _ = stats.spearmanr(y_shuf, y_pred)
        perm_rs.append(r_p)

    perm_rs = np.array(perm_rs)
    p_val   = (np.sum(perm_rs >= true_r) + 1) / (N_PERM + 1)
    return p_val

# ── Group-level test ───────────────────────────────────────────────────────────
def group_test(r_vals):
    r_clipped = np.clip(r_vals, -0.9999, 0.9999)
    z_vals    = np.arctanh(r_clipped)
    n         = len(z_vals)
    mean_r    = float(np.tanh(np.mean(z_vals)))
    sem_z     = float(np.std(z_vals, ddof=1) / np.sqrt(n))
    t, p_t    = stats.ttest_1samp(z_vals, 0)
    w, p_w    = stats.wilcoxon(z_vals) if n >= 5 else (np.nan, np.nan)
    return mean_r, sem_z, float(t), float(p_t), \
           (float(w) if not np.isnan(w) else np.nan), \
           (float(p_w) if not np.isnan(p_w) else np.nan)

def sig_stars(p):
    return "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else "ns"))

# ════════════════════════════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════════════════════════════

summary_rows = []
persubj_rows = []
coef_rows    = []

for epoch_name, dv_list in EPOCHS.items():
    for outcome_col in OUTCOMES:

        print(f"\n{'='*65}")
        print(f"  Epoch: {epoch_name}  |  Outcome: {outcome_col}")

        wide         = build_band_matrix(df, dv_list, outcome_col)
        feature_cols = get_feature_cols(wide, dv_list)

        print(f"  Words total : {len(wide)}  |  Features: {len(feature_cols)}")
        print(f"  Feature names: {feature_cols}")

        subj_r_vals = []

        for subj, sg in wide.groupby("subject"):
            n_words = len(sg)
            if n_words < MIN_WORDS:
                print(f"    Skipping {subj} (n={n_words} < {MIN_WORDS})")
                continue

            X = sg[feature_cols].values
            y = sg[outcome_col].values

            r, alpha, l1, coefs, n_nonzero = loo_elasticnet(X, y)
            p_perm = permutation_test(X, y, r, alpha, l1)

            print(f"    {subj:10s}  n={n_words:3d}  "
                  f"alpha={alpha:.4f}  l1={l1:.2f}  "
                  f"nonzero={n_nonzero:2d}  "
                  f"r={r:+.3f}  perm_p={p_perm:.3f}")

            subj_r_vals.append(r)
            persubj_rows.append({
                "epoch":      epoch_name,
                "outcome":    outcome_col,
                "subject":    subj,
                "n_words":    n_words,
                "loo_r":      round(r, 4),
                "perm_p":     round(p_perm, 4),
                "best_alpha": round(alpha, 5),
                "best_l1":    round(l1, 3),
                "n_nonzero":  n_nonzero,
            })

            for feat, coef in zip(feature_cols, coefs):
                coef_rows.append({
                    "epoch":   epoch_name,
                    "outcome": outcome_col,
                    "subject": subj,
                    "feature": feat,
                    "coef":    coef,
                })

        # Group test
        if len(subj_r_vals) < 3:
            print("  Not enough subjects for group test")
            continue

        mean_r, sem_z, t, p_t, w, p_w = group_test(np.array(subj_r_vals))
        print(f"\n  GROUP: n={len(subj_r_vals)}  mean_r={mean_r:+.4f}  "
              f"t={t:+.3f}  p={p_t:.4f}  {sig_stars(p_t)}")

        summary_rows.append({
            "epoch":    epoch_name,
            "outcome":  outcome_col,
            "n_subj":   len(subj_r_vals),
            "mean_r":   round(mean_r, 4),
            "sem_z":    round(sem_z, 4),
            "t":        round(t, 4),
            "p_ttest":  round(p_t, 4),
            "w_stat":   round(w, 4) if not np.isnan(w) else np.nan,
            "p_wilcox": round(p_w, 4) if not np.isnan(p_w) else np.nan,
        })

# ── FDR across conditions ──────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows)
if len(summary_df) > 0:
    _, fdr_p, _, _ = multipletests(summary_df["p_ttest"], method="fdr_bh")
    summary_df["p_fdr"] = fdr_p.round(4)
    summary_df["sig"]   = summary_df["p_fdr"].apply(sig_stars)

# ── Save ───────────────────────────────────────────────────────────────────────
summary_df.to_csv("elasticnet_band_summary.csv", index=False)
pd.DataFrame(persubj_rows).to_csv("elasticnet_band_persubj.csv", index=False)

coef_df = pd.DataFrame(coef_rows)
if len(coef_df) > 0:
    coef_mean = (
        coef_df.groupby(["epoch", "outcome", "feature"])["coef"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    coef_mean.to_csv("elasticnet_band_coefficients.csv", index=False)

print("\n\n══ SUMMARY ═══════════════════════════════════════════════════")
print(summary_df.to_string(index=False))
print("\nSaved:")
print("  elasticnet_band_summary.csv")
print("  elasticnet_band_persubj.csv")
print("  elasticnet_band_coefficients.csv")

Loading data ...
  Subjects : 36
  Rows     : 28836
  Band counts:
theta         8010
beta          6408
high_gamma    6408
low_gamma     4806
alpha         3204

  Epoch: encoding  |  Outcome: SP_clustering_from_prev
  Words total : 401  |  Features: 20
  Feature names: ['LHP_encoding_frac_ssl_alpha', 'LHP_encoding_frac_ssl_beta', 'LHP_encoding_frac_ssl_high_gamma', 'LHP_encoding_frac_ssl_low_gamma', 'LHP_encoding_frac_ssl_theta', 'LHP_encoding_osc_ssl_alpha', 'LHP_encoding_osc_ssl_beta', 'LHP_encoding_osc_ssl_high_gamma', 'LHP_encoding_osc_ssl_low_gamma', 'LHP_encoding_osc_ssl_theta', 'RHP_encoding_frac_ssl_alpha', 'RHP_encoding_frac_ssl_beta', 'RHP_encoding_frac_ssl_high_gamma', 'RHP_encoding_frac_ssl_low_gamma', 'RHP_encoding_frac_ssl_theta', 'RHP_encoding_osc_ssl_alpha', 'RHP_encoding_osc_ssl_beta', 'RHP_encoding_osc_ssl_high_gamma', 'RHP_encoding_osc_ssl_low_gamma', 'RHP_encoding_osc_ssl_theta']
    R1501J      n= 57  alpha=100.0000  l1=0.10  nonzero= 0  r=-1.000  perm_p=0.638
  